# SPATIAL INTELLIGENCE - PART 1

In [ ]:
# This cell is not needed if you have pip installed topologicpy
# import sys
# sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 1. Import the needed libraries

In [ ]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 2. Check the TopologicPy Version

In [ ]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [ ]:
renderer = "vscode"

## 4. Import the main House OBJ file

In [ ]:
House = Topology.ByOBJPath(r"D:\3rd sem\GRAPH MACHINE LEARNING\ASSIGNMENTS\1\complex house.obj", selfMerge=False)
print("House is a list")
print(House)

## 5. Convert List to Cluster


In [ ]:
# Convert House list to a Cluster first
house_cluster = Cluster.ByTopologies(House)
house = CellComplex.ByFacesCluster(house_cluster)
print("House is a cell complex")
print(len(House))


## 6. Import Doors and Windows OBJ file

In [ ]:
door = Topology.ByOBJPath(r"D:\3rd sem\GRAPH MACHINE LEARNING\ASSIGNMENTS\1\complex_door.obj", selfMerge=False)
door_c = Cluster.ByTopologies(door)
door = Topology.Faces(door_c)
print(f"Number of doors in cellcomplex: {len(door) if door else 0}")



window = Topology.ByOBJPath(r"D:\3rd sem\GRAPH MACHINE LEARNING\ASSIGNMENTS\1\complex_window.obj", selfMerge=False)
window_c = Cluster.ByTopologies(window)
window = Topology.Faces(window_c)
print(f"Number of windows in cellcomplex: {len(window) if window else 0}")

## 7. Cell Extraction and Color Tagging

In [ ]:
cells = []
selectors = []

# Define color map using RGB tuples (0-1 range)
color_map = {
    "Living_Room": "red",
    "Kitchen": "yellow",
    "Dining_Room": "pink",
    "Bedroom": "blue",
    "Bathroom": "purple",
    "Exterior_Corridor": "grey",
    "Interior_Corridor": "cyan",
    "Door": "brown",
    "Window": "light cyan"
}

# Extract cells from ORIGINAL House geometry (before apertures) to get room names
print(f"Processing {len(House)} original geometries...\n")

for obj in House:
    # Get the dictionary from the original object
    d = Topology.Dictionary(obj)
    name = Dictionary.ValueAtKey(d, "name") if d else None
    
    # Get color from color_map (default to gray if not found)
    color = color_map.get(name, [0.83, 0.83, 0.83])
    
    print(f"Room: {name} → color: {color}")
    
    # Extract cells from this object
    merged = Topology.SelfMerge(obj)
    cells_from_obj = Topology.Cells(merged) or []
    
    # Tag each cell with room name and color
    for cell in cells_from_obj:
        d_tagged = Dictionary.ByKeysValues(["name", "color"], [name, color])
        c = Topology.SetDictionary(cell, d_tagged)
        cells.append(c)
        selector = Topology.InternalVertex(cell)
        s = Topology.SetDictionary(selector, d_tagged)
        selectors.append(s)

# Add doors with their color
print(f"\nProcessing {len(door) if door else 0} doors...")
for door_face in (door or []):
    d_door = Dictionary.ByKeysValues(["name", "color"], ["Door", "brown"])
    door_tagged = Topology.SetDictionary(door_face, d_door)
    cells.append(door_tagged)

# Add windows with their color
print(f"Processing {len(window) if window else 0} windows...")
for window_face in (window or []):
    d_window = Dictionary.ByKeysValues(["name", "color"], ["Window", "light cyan"])
    window_tagged = Topology.SetDictionary(window_face, d_window)
    cells.append(window_tagged)

print(f"\nTotal cells extracted: {len(cells)} (rooms + doors + windows)")
print(len(cells))

In [ ]:
Topology.Show(cells,
               faceColorKey="color", backgroundColor="white", renderer=renderer)

## 8. Creating CellComplex

In [ ]:
cc= Topology.AddApertures(house, door, exclusive=False, subTopologyType="face")
cc= Topology.AddApertures(house, window, exclusive=False, subTopologyType="face")  

## 9. Aperture Data Extraction & Connectivity Graph Creation

In [ ]:
# door and window are already Face objects from OBJ files
aperture_faces = (door or []) + (window or [])

print(f"Found {len(aperture_faces)} aperture faces")
print(f"  Doors: {len(door) if door else 0}")
print(f"  Windows: {len(window) if window else 0}")

In [ ]:
aperture_data = []

for face in aperture_faces:
    f_centroid = Topology.Centroid(face)
    d = Topology.Dictionary(face)
    
    f_color = Dictionary.ValueAtKey(d, "color") if d else "brown"  # default
    f_type = Dictionary.ValueAtKey(d, "name") if d else "Aperture"
    
    aperture_data.append((f_centroid, f_color, f_type))

In [ ]:
g = Graph.ByTopology(
    cc,
    direct=False,
    viaSharedApertures=True,
    toExteriorApertures=True
)
print("Graph vertices:", len(Graph.Vertices(g)))
print("Edges:", len(Graph.Edges(g)))

## 10. Graph Vertex and Edge Processing

In [ ]:
# --- Color map ---
color_hex = {
    "red":"#FF0000","blue":"#0000FF","yellow":"#FFFF00",
    "pink":"#FF1493","purple":"#800080","cyan":"#00FFFF",
    "grey":"#808080","brown":"#8B4513","light cyan":"#E0FFFF"
}

# --- Cell data ---
cell_data = [
    (
        Topology.Centroid(c),
        Cell.SurfaceArea(c),
        color_hex.get(
            Dictionary.ValueAtKey(Topology.Dictionary(c), "color") if Topology.Dictionary(c) else "grey",
            "#808080"
        )
    )
    for c in cells
]

areas = [a for _, a, _ in cell_data]
min_a, max_a = min(areas), max(areas)

# --- Update vertices ---
updated_verts = []
for i, v in enumerate(Graph.Vertices(g)):
    vc = Topology.Centroid(v)
    _, a, col = min(cell_data, key=lambda c: Vertex.Distance(vc, c[0]))

    size = 8 + 20 * (((a - min_a) / (max_a - min_a + 1e-6)) ** 0.5)

    d = Topology.Dictionary(v) or Dictionary.ByKeysValues([], [])
    for k, val in [("size", size), ("color", col)]:
        d = Dictionary.SetValueAtKey(d, k, val)

    updated_verts.append(Topology.SetDictionary(v, d))

# --- Apertures (doors + windows) ---
aperture_verts = [
    Topology.SetDictionary(
        Vertex.ByCoordinates(c.X, c.Y, c.Z),
        Dictionary.ByKeysValues(["size","color"], [s, col])
    )
    for faces, s, col in [(door,5,"#8B4513"), (window,4,"#E0FFFF")]
    for f in (faces or [])
    for c in [Topology.Centroid(f)]
]

# --- Rebuild graph ---
all_verts = updated_verts + aperture_verts
g = Graph.ByVerticesEdges(all_verts, Graph.Edges(g))

# --- Edge styling ---
for e in Graph.Edges(g):
    Topology.SetDictionary(e, Dictionary.ByKeysValues(["width","color"], [10,"black"]))

## 11.Visualisation

In [ ]:
Topology.Show(g, cc, door, window, 
               vertexSizeKey="size", vertexColorKey="color", backgroundColor="white", renderer=renderer)